# 观测拟合与房价

先手算一个比例系数，再改变第三套房的观测价格。房价数据与模型都是教学假设。

依赖：NumPy、Matplotlib。按顺序运行，先在纸上预测再核对；图内使用英文标签。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 三观测、一个系数

先求 a^Tb/a^Ta，再核对残差是否垂直 a。


In [ ]:
a = np.array([1.,2.,3.])
b = np.array([1.,2.,2.])
c = (a @ b)/(a @ a)
residual = b-c*a
print('系数:',c,'预测:',c*a,'残差:',residual)
assert np.allclose(c,11/14)
assert np.allclose(a @ residual,0)


## 三套房、两项系数

第三套房若从 175 万改成 180 万，新拟合是否还让前两套房精确成立？先预测，再运行。


In [ ]:
A = np.array([[80.,1.],[100.,2.],[90.,.5]])
old_coefficients = np.array([2.,-10.])

def fit_houses(third_price=180.):
    prices = np.array([150.,180.,third_price])
    coefficients, _, rank, _ = np.linalg.lstsq(A,prices,rcond=None)
    predictions = A @ coefficients
    residual = prices-predictions
    print('面积、距离系数:',coefficients,'秩:',rank)
    print('预测:',predictions,'残差:',residual)
    print('A^T r:',A.T @ residual)
    assert np.allclose(A.T @ residual,0,atol=1e-8)
    assert np.linalg.norm(residual) <= np.linalg.norm(prices-A @ old_coefficients)+1e-10
    positions = np.arange(3)
    fig,axes = plt.subplots(1,2,figsize=(10,4),dpi=150)
    axes[0].bar(positions-.18,prices,width=.36,color='#2878b5',label='Observed')
    axes[0].bar(positions+.18,predictions,width=.36,color='#d97922',label='Fitted')
    axes[0].set(xticks=positions,xticklabels=['House A','House B','House C'],
                ylabel='price (10,000 yuan)',title='One fit for all three houses',ylim=(0,210))
    axes[0].legend()
    axes[1].bar(positions,residual,color='#359969')
    axes[1].axhline(0,color='0.4',lw=.8)
    axes[1].set(xticks=positions,xticklabels=['A','B','C'],
                ylabel='observed minus fitted',title='Residuals',ylim=(-10,10))
    fig.tight_layout()
    plt.show()
    return coefficients

fit_houses()


## 改一个观测

恢复到 175 时，还会有残差吗？滑块范围内所有绘图数据都能完整显示。


In [ ]:
fit_houses(third_price=175.)
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('直接修改 fit_houses(third_price=...)。')
else:
    display(widgets.interactive(fit_houses,
        third_price=widgets.FloatSlider(value=180,min=165,max=185,step=1,
                                       continuous_update=False)))


## 合上实验，自己解释

指出已知量、未知量、单位与计算目的。用纸上推导解释观察结果，再改变一个参数检查；有限次计算不是一般结论的证明。
